# BiLSTM + Attention Visualization
Loads a saved BiLSTM-attention checkpoint from Drive, runs inference on tweets, and visualizes attention weights.

In [ ]:
!pip install -q matplotlib seaborn pandas torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# --- Paths ---
DATA_PATH        = "/content/drive/MyDrive/CS4248/"
MODEL_PATH       = os.path.join(DATA_PATH, "lstm/best_model.pt")
TOKENIZER_PATH   = os.path.join(DATA_PATH, "lstm/tokenizer.pkl")

LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_LEN     = 64

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

## Model definition
Identical to `lstm_attention_colab.ipynb` — must match the saved checkpoint exactly.

In [ ]:
class CustomWordTokenizer:
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(str(text).lower().split())
        for word, _ in counter.most_common(self.max_vocab_size - 2):
            self.word2idx[word] = self.vocab_size
            self.idx2word[self.vocab_size] = word
            self.vocab_size += 1

    def encode(self, text, max_length):
        words = str(text).lower().split()
        ids = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in words]
        if len(ids) > max_length:
            ids = ids[:max_length]
        else:
            ids = ids + [self.word2idx['<PAD>']] * (max_length - len(ids))
        return ids, words[:max_length]  # also return the actual tokens for display


class LSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=3,
                 num_layers=2, bidirectional=True, dropout=0.5, num_heads=8,
                 pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()
        if pretrained_embeddings is not None:
            embedding_dim = pretrained_embeddings.shape[1]
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=bidirectional, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_output_dim, num_heads=num_heads,
            batch_first=True, dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_output_dim, num_classes)

    def forward(self, input_ids, return_attention=False):
        pad_mask = (input_ids == 0)
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)

        attn_out, attn_weights = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=pad_mask,
            need_weights=True,
            average_attn_weights=True,
        )

        token_scores = attn_weights.sum(dim=1)
        token_scores = token_scores.masked_fill(pad_mask, 0.0)
        token_scores = token_scores / token_scores.sum(dim=1, keepdim=True).clamp(min=1e-9)

        context = (token_scores.unsqueeze(-1) * lstm_out).sum(dim=1)
        out = self.dropout(context)
        out = self.fc(out)

        if return_attention:
            return out, attn_weights, token_scores
        return out

## Load tokenizer and model

In [ ]:
print("Loading tokenizer...")
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)
print(f"Vocab size: {tokenizer.vocab_size}")

# --- Instantiate model (must match checkpoint hyperparameters) ---
GLOVE_DIM  = 200
HIDDEN_DIM = 128
NUM_HEADS  = 8
NUM_LAYERS = 1
DROPOUT    = 0.5

dummy_emb = torch.zeros(tokenizer.vocab_size, GLOVE_DIM)
model = LSTMAttentionClassifier(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=HIDDEN_DIM,
    num_classes=3,
    num_layers=NUM_LAYERS,
    bidirectional=True,
    dropout=DROPOUT,
    num_heads=NUM_HEADS,
    pretrained_embeddings=dummy_emb,
    freeze_embeddings=False,
)

print(f"Loading checkpoint from {MODEL_PATH}")
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)
model = model.to(DEVICE)
model.eval()
print("Model loaded.")

## Visualization helpers

In [ ]:
def predict_and_get_attention(text):
    """Run inference on a single tweet and return prediction, probs, attention matrix, token scores, and tokens."""
    ids, tokens = tokenizer.encode(text, MAX_LEN)
    # Keep only non-padding tokens for display
    seq_len = min(len(text.split()), MAX_LEN)
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits, attn_weights, token_scores = model(input_ids, return_attention=True)

    probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    pred  = probs.argmax()

    # Trim to actual sequence length (drop padding)
    attn_matrix  = attn_weights[0, :seq_len, :seq_len].cpu().numpy()  # (seq_len, seq_len)
    token_imp    = token_scores[0, :seq_len].cpu().numpy()             # (seq_len,)

    return pred, probs, attn_matrix, token_imp, tokens[:seq_len]


def plot_attention(tweet, figsize=(14, 5)):
    """Plot token importance bar chart and full attention heatmap side by side."""
    pred, probs, attn_matrix, token_imp, tokens = predict_and_get_attention(tweet)

    label_str = LABEL_NAMES[pred]
    conf      = probs[pred]
    title     = f'Prediction: {label_str}  (conf {conf:.2%})   |   neg {probs[0]:.2%}  neu {probs[1]:.2%}  pos {probs[2]:.2%}'

    fig, axes = plt.subplots(1, 2, figsize=figsize,
                             gridspec_kw={'width_ratios': [1, 2]})
    fig.suptitle(f'"{tweet}"\n{title}', fontsize=10, wrap=True)

    # --- Left: token importance bar chart ---
    ax = axes[0]
    colors = plt.cm.RdYlGn(token_imp / token_imp.max() if token_imp.max() > 0 else token_imp)
    ax.barh(range(len(tokens)), token_imp, color=colors)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Token importance score")
    ax.set_title("Token importance")

    # --- Right: full attention heatmap (query × key) ---
    ax2 = axes[1]
    sns.heatmap(
        attn_matrix,
        xticklabels=tokens,
        yticklabels=tokens,
        cmap="YlOrRd",
        ax=ax2,
        linewidths=0.3,
        linecolor='lightgrey',
    )
    ax2.set_title("Attention matrix (query × key)")
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=8)

    plt.tight_layout()
    plt.show()
    return pred, probs, tokens, token_imp


def plot_top_tokens(tweet, top_n=5):
    """Print and bar-plot the top-N most attended tokens."""
    pred, probs, _, token_imp, tokens = predict_and_get_attention(tweet)
    top_idx    = token_imp.argsort()[::-1][:top_n]
    top_tokens = [tokens[i] for i in top_idx]
    top_scores = token_imp[top_idx]

    print(f"Tweet : {tweet}")
    print(f"Pred  : {LABEL_NAMES[pred]}  ({probs[pred]:.2%})")
    print(f"Top {top_n} tokens:")
    for tok, score in zip(top_tokens, top_scores):
        print(f"  {tok:<20s}  {score:.4f}")

    plt.figure(figsize=(6, 3))
    plt.barh(top_tokens[::-1], top_scores[::-1], color='steelblue')
    plt.xlabel("Importance score")
    plt.title(f'Top {top_n} tokens — "{tweet[:60]}"')
    plt.tight_layout()
    plt.show()

## Run visualizations

In [ ]:
# =============================================================================
# ANGLE 1: BiLSTM+Attention correct, plain BiLSTM wrong
# These show what the attention mechanism contributes over plain BiLSTM
# =============================================================================

# BiLSTM predicted pos, attention correctly predicted neg
# Expect attention to focus on "ban"
plot_attention("Hard to be a free speech platform in a state that wants to ban free speech")

# BiLSTM predicted neg, attention correctly predicted pos
# Expect attention to find "powerup" over misleading "wow" context
plot_attention("Wow, steak & eggs with coffee in the morning really feels like a powerup!")

# BiLSTM predicted neu, attention correctly predicted neg
# Short tweet — attention should lock onto "contempt" and "galling"
plot_attention("The contempt he showed for America is galling")

# BiLSTM predicted neu, attention correctly predicted neg
# Politically charged — attention should pick up "falls"
plot_attention("If America falls, nothing else matters")

In [ ]:
# =============================================================================
# ANGLE 2: RoBERTa correct, BiLSTM+Attention wrong
# These show where word-level attention still fails vs subword contextual models
# =============================================================================

# BiLSTM predicted pos, RoBERTa correctly predicted neg
# BiLSTM likely fires on "very" as a positive intensifier
plot_attention("Very misleading ...")

# BiLSTM predicted pos, RoBERTa correctly predicted neg
# BiLSTM likely fires on "Exactly" as a positive signal
plot_attention("Exactly. Total madness!")

# BiLSTM predicted pos, RoBERTa correctly predicted neg
# BiLSTM may fire on "America" as positive; RoBERTa reads the full clause
plot_attention("The contempt he showed for America is galling")

# BiLSTM predicted neg, RoBERTa correctly predicted pos
# "China" or "accomplishment" may be negatively coloured in BiLSTM training data
plot_attention("I would like to second that congratulations. Outstanding accomplishment by China!")

# BiLSTM predicted neu, RoBERTa correctly predicted pos
# "joys" should be a strong signal — suggests attention is spreading weight too flat
plot_attention("There are many small joys as you explore the car & more to come via software updates")

In [ ]:
# =============================================================================
# Top-N token importance for selected tweets
# =============================================================================

plot_top_tokens("Very misleading ...", top_n=5)
plot_top_tokens("Exactly. Total madness!", top_n=5)
plot_top_tokens("Hard to be a free speech platform in a state that wants to ban free speech", top_n=5)